In [9]:

# ==========================================
# 0. CÀI ĐẶT THƯ VIỆN
# ==========================================
!pip install -q transformers pandas scikit-learn

import os
import json
import logging
import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from google.colab import drive

# ==========================================
# 1. KẾT NỐI GOOGLE DRIVE
# ==========================================
print("Đang kết nối với Google Drive...")
drive.mount('/content/drive')

drive_path = '/content/drive/MyDrive/train-KLCN'
os.makedirs(drive_path, exist_ok=True)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = os.path.join(
    drive_path, f"output_roberta_{timestamp}"
)
os.makedirs(output_dir, exist_ok=True)

log_file = os.path.join(output_dir, 'training_log.txt')

print(f"Thư mục kết quả: {output_dir}")
print(f"File log: {log_file}")

# ==========================================
# 2. CẤU HÌNH LOGGING
# ==========================================
logger = logging.getLogger("roberta_training")
logger.setLevel(logging.INFO)
logger.propagate = False

# Xóa handler cũ khi chạy lại cell
for handler in logger.handlers[:]:
    handler.close()
    logger.removeHandler(handler)

file_handler = logging.FileHandler(
    log_file, mode='w', encoding='utf-8'
)
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(
    logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s'
    )
)

console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(
    logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s'
    )
)

logger.addHandler(file_handler)
logger.addHandler(console_handler)

def flush_logs():
    for handler in logger.handlers:
        handler.flush()

logger.info("Bắt đầu phiên huấn luyện RoBERTa.")
logger.info(f"Thư mục lưu kết quả: {output_dir}")
flush_logs()

# ==========================================
# 3. ĐỌC VÀ TIỀN XỬ LÝ DATASET EMOBANK
# ==========================================
logger.info("Đang đọc dữ liệu EmoBank...")

file_path = os.path.join(drive_path, 'emobank.csv')

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"Không tìm thấy file: {file_path}. "
        "Hãy tải emobank.csv lên thư mục train-KLCN trên Drive."
    )

df = pd.read_csv(file_path)

required_columns = ['text', 'V', 'A']
missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Dataset thiếu các cột: {missing_columns}"
    )

df = df[required_columns].copy()
df = df.dropna(subset=['text', 'V', 'A'])

df['V'] = pd.to_numeric(df['V'], errors='coerce')
df['A'] = pd.to_numeric(df['A'], errors='coerce')
df = df.dropna(subset=['V', 'A'])
df = df[df['text'].astype(str).str.strip() != '']

if len(df) < 2:
    raise ValueError("Dataset không đủ dữ liệu để huấn luyện.")

# Chuẩn hóa V và A từ thang [1, 5] sang [-1, 1]
df['V_norm'] = (df['V'] - 3.0) / 2.0
df['A_norm'] = (df['A'] - 3.0) / 2.0

logger.info(f"Tổng số mẫu hợp lệ: {len(df)}")
logger.info("Đã chuẩn hóa V và A về thang [-1, 1].")
flush_logs()

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].astype(str).tolist(),
    df[['V_norm', 'A_norm']].values.astype(np.float32).tolist(),
    test_size=0.1,
    random_state=42
)

logger.info(f"Số mẫu huấn luyện: {len(train_texts)}")
logger.info(f"Số mẫu validation: {len(val_texts)}")
flush_logs()

# ==========================================
# 4. DATASET VÀ DATALOADER
# ==========================================
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])

        label = torch.tensor(
            self.labels[idx],
            dtype=torch.float32
        )

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': label
        }


logger.info("Đang tải tokenizer RoBERTa...")
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

train_dataset = EmotionDataset(
    train_texts, train_labels, tokenizer
)
val_dataset = EmotionDataset(
    val_texts, val_labels, tokenizer
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

logger.info("Đã tạo Dataset và DataLoader.")
flush_logs()

# ==========================================
# 5. XÂY DỰNG MÔ HÌNH ROBERTA REGRESSION
# ==========================================
class RobertaForVARegression(nn.Module):
    def __init__(self, checkpoint='roberta-base'):
        super().__init__()

        self.roberta = RobertaModel.from_pretrained(checkpoint)
        self.dropout = nn.Dropout(0.3)
        self.regressor = nn.Linear(
            self.roberta.config.hidden_size, 2
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Lấy biểu diễn của token đầu tiên <s>
        pooled_output = outputs.last_hidden_state[:, 0, :]
        dropped_output = self.dropout(pooled_output)

        return self.regressor(dropped_output)


device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

logger.info(f"Thiết bị huấn luyện: {device}")

model = RobertaForVARegression().to(device)

criterion = nn.MSELoss()
optimizer = AdamW(model.parameters(), lr=2e-5)

# ==========================================
# 6. HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH
# ==========================================
epochs = 3
metrics_history = []

logger.info("BẮT ĐẦU HUẤN LUYỆN")
flush_logs()

for epoch in range(epochs):
    # --------------------------------------
    # Huấn luyện
    # --------------------------------------
    model.train()
    total_train_loss = 0.0
    train_sample_count = 0

    logger.info(
        f"========== EPOCH {epoch + 1}/{epochs} =========="
    )
    flush_logs()

    for step, batch in enumerate(train_loader, start=1):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_train_loss += loss.item() * batch_size
        train_sample_count += batch_size

        if (
            step == 1
            or step % 100 == 0
            or step == len(train_loader)
        ):
            avg_train_loss = (
                total_train_loss / train_sample_count
            )

            logger.info(
                f"Epoch [{epoch + 1}/{epochs}] | "
                f"Step [{step}/{len(train_loader)}] | "
                f"Train MSE: {avg_train_loss:.6f}"
            )
            flush_logs()

    avg_train_loss = (
        total_train_loss / train_sample_count
    )

    # --------------------------------------
    # Validation
    # --------------------------------------
    model.eval()

    total_val_loss = 0.0
    total_abs_error = 0.0
    total_squared_error = 0.0
    val_sample_count = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask)
            batch_size = labels.size(0)

            total_val_loss += (
                criterion(outputs, labels).item() * batch_size
            )

            total_abs_error += torch.abs(
                outputs - labels
            ).sum().item()

            total_squared_error += (
                (outputs - labels) ** 2
            ).sum().item()

            val_sample_count += batch_size

    avg_val_loss = total_val_loss / val_sample_count

    # MAE và RMSE tính trên từng giá trị V/A
    avg_mae = total_abs_error / (val_sample_count * 2)

    avg_rmse = float(np.sqrt(
        total_squared_error / (val_sample_count * 2)
    ))

    logger.info(f"Epoch {epoch + 1} hoàn tất.")
    logger.info(f"Train MSE: {avg_train_loss:.6f}")
    logger.info(f"Validation MSE: {avg_val_loss:.6f}")
    logger.info(f"Validation MAE: {avg_mae:.6f}")
    logger.info(f"Validation RMSE: {avg_rmse:.6f}")
    flush_logs()

    metrics_history.append({
        'epoch': epoch + 1,
        'train_mse': avg_train_loss,
        'val_mse': avg_val_loss,
        'val_mae': avg_mae,
        'val_rmse': avg_rmse
    })

    # Lưu metrics sau mỗi epoch
    metrics_file = os.path.join(
        output_dir, 'training_metrics.json'
    )

    with open(metrics_file, 'w', encoding='utf-8') as f:
        json.dump(
            metrics_history, f,
            indent=4, ensure_ascii=False
        )

# ==========================================
# 7. LƯU MÔ HÌNH VÀ TOKENIZER
# ==========================================
logger.info("Đang lưu mô hình...")
flush_logs()

model_save_path = os.path.join(
    output_dir, 'roberta_va_regression.pt'
)
torch.save(model.state_dict(), model_save_path)

tokenizer_dir = os.path.join(output_dir, 'tokenizer')
tokenizer.save_pretrained(tokenizer_dir)

logger.info(f"Đã lưu trọng số mô hình: {model_save_path}")
logger.info(f"Đã lưu tokenizer: {tokenizer_dir}")
logger.info(f"Đã lưu metrics: {metrics_file}")
logger.info("HUẤN LUYỆN HOÀN TẤT.")
logger.info(f"Tất cả kết quả nằm tại: {output_dir}")
flush_logs()

# Đóng file log sau khi đã ghi xong
for handler in logger.handlers[:]:
    handler.flush()
    handler.close()
    logger.removeHandler(handler)

print("\nHuấn luyện hoàn tất!")
print(f"Thư mục kết quả: {output_dir}")
print(f"File log: {log_file}")
print(f"File metrics: {metrics_file}")
print(f"File mô hình: {model_save_path}")

Đang kết nối với Google Drive...


2026-10-10 14:52:52,616 - INFO - Bắt đầu phiên huấn luyện RoBERTa.
2026-10-10 14:52:52,618 - INFO - Thư mục lưu kết quả: /content/drive/MyDrive/train-KLCN/output_roberta_20261010_145252
2026-10-10 14:52:52,619 - INFO - Đang đọc dữ liệu EmoBank...
2026-10-10 14:52:52,660 - INFO - Tổng số mẫu hợp lệ: 10061
2026-10-10 14:52:52,662 - INFO - Đã chuẩn hóa V và A về thang [-1, 1].
2026-10-10 14:52:52,671 - INFO - Số mẫu huấn luyện: 9054
2026-10-10 14:52:52,673 - INFO - Số mẫu validation: 1007
2026-10-10 14:52:52,675 - INFO - Đang tải tokenizer RoBERTa...


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thư mục kết quả: /content/drive/MyDrive/train-KLCN/output_roberta_20261010_145252
File log: /content/drive/MyDrive/train-KLCN/output_roberta_20261010_145252/training_log.txt


2026-10-10 14:52:54,360 - INFO - Đã tạo Dataset và DataLoader.
2026-10-10 14:52:54,363 - INFO - Thiết bị huấn luyện: cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
2026-10-10 14:52:55,110 - INFO - BẮT ĐẦU HUẤN LUYỆN
2026-10-10 14:52:55,114 - INFO - ========== EPOCH 1/3 ==========
2026-10-10 14:52:55,514 - INFO - Epoch [1/3] | Step [1/566] | Train MSE: 0.062304
2026-10-10 14:53:31,178 - INFO - Epoch [1/3] | Step [100/566] | Train MSE: 0.039054
2026-10-10 14:54:08,241 - INFO -


Huấn luyện hoàn tất!
Thư mục kết quả: /content/drive/MyDrive/train-KLCN/output_roberta_20261010_145252
File log: /content/drive/MyDrive/train-KLCN/output_roberta_20261010_145252/training_log.txt
File metrics: /content/drive/MyDrive/train-KLCN/output_roberta_20261010_145252/training_metrics.json
File mô hình: /content/drive/MyDrive/train-KLCN/output_roberta_20261010_145252/roberta_va_regression.pt


In [3]:
import torch
import torch.nn as nn
from transformers import RobertaTokenizer, RobertaModel

# ==========================================
# 1. ĐỊNH NGHĨA LẠI KIẾN TRÚC MÔ HÌNH
# ==========================================
class RobertaForVARegression(nn.Module):
    def __init__(self, checkpoint='roberta-base'):
        super(RobertaForVARegression, self).__init__()
        self.roberta = RobertaModel.from_pretrained(checkpoint)
        self.regressor = nn.Linear(self.roberta.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        return self.regressor(pooled_output)

# ==========================================
# 2. KHỞI TẠO MÔ HÌNH VÀ TẢI TRỌNG SỐ
# ==========================================
print("Đang tải mô hình để test...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

model = RobertaForVARegression().to(device)

# Đường dẫn tới file bạn vừa train xong trên Drive
# (Nếu tải về máy tính thì đổi đường dẫn này thành đường dẫn file local)
model_path = '/content/drive/MyDrive/train-KLCN/roberta_va_regression.pt'

# Load weights
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval() # Chuyển sang chế độ suy luận

# ==========================================
# 3. HÀM DỰ ĐOÁN
# ==========================================
def predict_mood(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding='max_length',
        max_length=128
    )

    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask)

    valence = outputs[0][0].item()
    arousal = outputs[0][1].item()

    return valence, arousal

# ==========================================
# 4. CHẠY TEST THỬ CÁC NGỮ CẢNH CẢM XÚC
# ==========================================
test_sentences = [
    "I feel so happy and energetic today! Let's go party!",  # Kỳ vọng: V cao, A cao (Nhạc Dance/Phim Hài)
    "I am completely exhausted, sad, and just want to cry.", # Kỳ vọng: V thấp, A thấp (Nhạc Ballad/Phim Drama buồn)
    "I'm just relaxing and chilling with a cup of tea.",     # Kỳ vọng: V cao, A thấp (Nhạc Lofi/Phim Chữa lành)
    "I am so angry and stressed right now! I hate this!",    # Kỳ vọng: V thấp, A cao (Nhạc Rock/Phim Hành động)
    "Just a normal day, nothing special happened."           # Kỳ vọng: V ~ 0, A ~ 0 (Trung tính)
]

print("\n--- KẾT QUẢ TEST MÔ HÌNH ---")
for sentence in test_sentences:
    v, a = predict_mood(sentence)
    print(f"Câu nhập: '{sentence}'")
    print(f" -> Tọa độ dự đoán: Valence = {v:.4f} | Arousal = {a:.4f}\n")

Đang tải mô hình để test...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- KẾT QUẢ TEST MÔ HÌNH ---
Câu nhập: 'I feel so happy and energetic today! Let's go party!'
 -> Tọa độ dự đoán: Valence = 0.4689 | Arousal = 0.5587

Câu nhập: 'I am completely exhausted, sad, and just want to cry.'
 -> Tọa độ dự đoán: Valence = -0.5589 | Arousal = 0.2810

Câu nhập: 'I'm just relaxing and chilling with a cup of tea.'
 -> Tọa độ dự đoán: Valence = 0.1237 | Arousal = -0.0390

Câu nhập: 'I am so angry and stressed right now! I hate this!'
 -> Tọa độ dự đoán: Valence = -0.5524 | Arousal = 0.6816

Câu nhập: 'Just a normal day, nothing special happened.'
 -> Tọa độ dự đoán: Valence = -0.0128 | Arousal = -0.0699



In [5]:
test_sentences = [
    "I miss the person I used to be, even though I know I can never go back to that time."
]

print("\n--- KẾT QUẢ TEST MÔ HÌNH ---")
for sentence in test_sentences:
    v, a = predict_mood(sentence)
    print(f"Câu nhập: '{sentence}'")
    print(f" -> Tọa độ dự đoán: Valence = {v:.4f} | Arousal = {a:.4f}\n")


--- KẾT QUẢ TEST MÔ HÌNH ---
Câu nhập: 'I miss the person I used to be, even though I know I can never go back to that time.'
 -> Tọa độ dự đoán: Valence = -0.2535 | Arousal = 0.0430

